# Ingenieria de variables Bolivia - Objetivo A

Este notebook genera variables predictivas para el dataset de Bolivia con foco en minimizar falsos positivos en compras presenciales. prepara `df_features`, define `feature_cols`, excluye columnas con leakage.

## 1) Configuración y carga

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


In [ ]:
candidate_paths = [Path('../data/bolivia_dataset.csv'), Path('data/bolivia_dataset.csv')]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('No se encontro bolivia_dataset.csv en data/.')

df = pd.read_csv(data_path, sep=';')
target_col = 'is_fraud'

bool_map = {'true': True, 'false': False, '1': True, '0': False, 1: True, 0: False, True: True, False: False}
for col in [target_col, 'is_international', 'approved']:
    if col in df.columns and df[col].dtype != bool:
        df[col] = df[col].map(lambda x: bool_map.get(str(x).lower(), x))

print(f'Ruta usada: {data_path.resolve()}')
print(f'Shape original: {df.shape}')
display(df.head())


## 2) Fecha transaccional y compras presenciales

`DE7_transmission_datetime` usa el formato ISO 8583 `MMDDhhmmss`. Como el dataset corresponde a 2025, se antepone ese año para crear `tx_datetime`. Compra presencial se define como `channel == 'POS'` y `DE60_pos_terminal_type == 'POS-ATTENDED'`.

In [ ]:
df_features = df.copy()

de7 = df_features['DE7_transmission_datetime'].astype(str).str.zfill(10)
df_features['tx_datetime'] = pd.to_datetime('2025' + de7, format='%Y%m%d%H%M%S', errors='coerce')
if df_features['tx_datetime'].isna().any():
    raise ValueError('Hay timestamps no parseables en DE7_transmission_datetime.')

df_features = df_features.sort_values(['tx_datetime', 'transaction_id']).reset_index(drop=True)

df_features['is_pos_attended'] = df_features['DE60_pos_terminal_type'].eq('POS-ATTENDED')
df_features['is_card_present_purchase'] = df_features['channel'].eq('POS') & df_features['is_pos_attended']
df_features['is_ecom'] = df_features['channel'].eq('ECOM')
df_features['is_atm'] = df_features['channel'].eq('ATM')
df_features['is_moto'] = df_features['channel'].eq('MOTO')
df_features['is_domestic'] = ~df_features['is_international'].astype(bool)
df_features['is_foreign_currency'] = df_features['currency_tx_alpha'].ne('BOB')
df_features['is_international_pos'] = df_features['is_card_present_purchase'] & df_features['is_international'].astype(bool)
df_features['requires_pin'] = df_features['DE52_pin_data_present'].eq('Y')
df_features['has_emv'] = df_features['DE55_emv_data_present'].eq('Y')

pos_eval_mask = df_features['is_card_present_purchase']

display(pd.DataFrame({
    'total_txn': [len(df_features)],
    'fraudes': [int(df_features[target_col].sum())],
    'fraud_rate_pct': [df_features[target_col].mean() * 100],
    'compras_presenciales': [int(pos_eval_mask.sum())],
    'fraudes_presenciales': [int(df_features.loc[pos_eval_mask, target_col].sum())],
    'fraud_rate_pos_pct': [df_features.loc[pos_eval_mask, target_col].mean() * 100],
}).round(3))


## 3) Variables de monto y anomalia

Para reducir falsos positivos, no basta marcar montos altos: se compara el monto contra el baseline y contra comportamiento historico previo del cliente, MCC y comercio. Las variables historicas no usan la fila actual.

In [ ]:
amount = pd.to_numeric(df_features['amount_usd'], errors='coerce')
baseline = pd.to_numeric(df_features['client_baseline_amount'], errors='coerce').replace(0, np.nan)

df_features['amount_usd_log1p'] = np.log1p(amount.clip(lower=0))
df_features['amount_vs_client_baseline'] = amount / baseline
df_features['amount_over_baseline_flag'] = df_features['amount_vs_client_baseline'].gt(1.5)

client_group = df_features.groupby('client_id', sort=False)['amount_usd']
df_features['client_txn_count_prev'] = client_group.cumcount()
df_features['client_amount_sum_prev'] = client_group.cumsum() - amount
df_features['client_amount_mean_prev'] = df_features['client_amount_sum_prev'] / df_features['client_txn_count_prev'].replace(0, np.nan)
df_features['client_amount_median_prev'] = client_group.transform(lambda s: s.expanding().median().shift(1))
df_features['client_amount_std_prev'] = client_group.transform(lambda s: s.expanding().std().shift(1))
df_features['amount_zscore_customer'] = (amount - df_features['client_amount_mean_prev']) / df_features['client_amount_std_prev'].replace(0, np.nan)
df_features['amount_vs_client_mean_prev'] = amount / df_features['client_amount_mean_prev'].replace(0, np.nan)
df_features['amount_vs_client_median_prev'] = amount / df_features['client_amount_median_prev'].replace(0, np.nan)

def add_prior_amount_mean(frame, group_cols, new_col):
    group = frame.groupby(group_cols, sort=False)['amount_usd']
    count_prev = group.cumcount()
    sum_prev = group.cumsum() - frame['amount_usd']
    frame[new_col] = sum_prev / count_prev.replace(0, np.nan)
    return frame

df_features = add_prior_amount_mean(df_features, ['DE18_merchant_category_code'], 'mcc_amount_mean_prev')
df_features = add_prior_amount_mean(df_features, ['DE42_card_acceptor_id'], 'merchant_amount_mean_prev')
df_features['amount_vs_mcc_mean_prev'] = amount / df_features['mcc_amount_mean_prev'].replace(0, np.nan)
df_features['amount_vs_merchant_mean_prev'] = amount / df_features['merchant_amount_mean_prev'].replace(0, np.nan)

amount_features = [
    'amount_usd_log1p', 'amount_vs_client_baseline', 'amount_over_baseline_flag',
    'client_txn_count_prev', 'client_amount_mean_prev', 'client_amount_median_prev',
    'client_amount_std_prev', 'amount_zscore_customer', 'amount_vs_client_mean_prev',
    'amount_vs_client_median_prev', 'amount_vs_mcc_mean_prev', 'amount_vs_merchant_mean_prev',
]
display(df_features[amount_features].describe(include='all').T)


## 4) Variables de velocidad transaccional

La velocidad captura compras repetidas o acumulados anormales. Las ventanas usan `closed='left'`, por lo que excluyen la transaccion actual.

In [ ]:
df_features['time_since_last_txn_min'] = (
    df_features.groupby('client_id', sort=False)['tx_datetime'].diff().dt.total_seconds().div(60)
)

windows = {'1h': '1h', '6h': '6h', '24h': '24h', '7d': '7d'}
for label in windows:
    df_features[f'txn_count_prev_{label}'] = 0.0
    df_features[f'amount_sum_prev_{label}'] = 0.0

for _, idx in df_features.groupby('client_id', sort=False).groups.items():
    sub = df_features.loc[idx, ['tx_datetime', 'amount_usd']].sort_values('tx_datetime')
    sub_ts = sub.set_index('tx_datetime')
    for label, window in windows.items():
        df_features.loc[sub.index, f'txn_count_prev_{label}'] = sub_ts['amount_usd'].rolling(window, closed='left').count().fillna(0).to_numpy()
        df_features.loc[sub.index, f'amount_sum_prev_{label}'] = sub_ts['amount_usd'].rolling(window, closed='left').sum().fillna(0).to_numpy()

velocity_features = ['time_since_last_txn_min'] + [f'txn_count_prev_{w}' for w in windows] + [f'amount_sum_prev_{w}' for w in windows]
display(df_features[velocity_features].describe().T.round(3))


## 5) Variables de novedad, contexto y riesgo histórico

Estas variables distinguen una compra presencial usual de combinaciones nuevas de cliente, comercio, MCC, moneda, pais o modo de entrada. Las tasas históricas de fraude se suavizan con el promedio global y solo usan eventos anteriores.

In [ ]:
novelty_specs = {
    'client_merchant': ['client_id', 'DE42_card_acceptor_id'],
    'client_mcc': ['client_id', 'DE18_merchant_category_code'],
    'client_currency': ['client_id', 'currency_tx_alpha'],
    'client_acquirer_country': ['client_id', 'DE19_acquirer_country_code'],
    'client_entry_mode': ['client_id', 'DE22_pos_entry_mode'],
}

novelty_features = []
for name, cols in novelty_specs.items():
    count_col = f'{name}_count_prev'
    first_col = f'is_first_{name}'
    df_features[count_col] = df_features.groupby(cols, sort=False).cumcount()
    df_features[first_col] = df_features[count_col].eq(0)
    novelty_features.extend([count_col, first_col])

global_prior = df_features[target_col].astype(int).mean()
alpha = 50
risk_rate_cols = ['channel', 'DE60_pos_terminal_type', 'DE18_merchant_category_code', 'DE22_pos_entry_mode', 'DE25_pos_condition_code', 'currency_tx_alpha']
risk_features = []

for col in risk_rate_cols:
    y = df_features[target_col].astype(int)
    group = df_features.groupby(col, sort=False)[target_col]
    count_prev = group.cumcount()
    fraud_sum_prev = group.cumsum().astype(float) - y
    new_col = f'{col}_fraud_rate_prev_smooth'
    df_features[new_col] = (fraud_sum_prev + alpha * global_prior) / (count_prev + alpha)
    risk_features.append(new_col)

display(df_features[novelty_features + risk_features].head(10))
display(df_features[novelty_features + risk_features].describe(include='all').T)


## 6) Variables temporales

Se agregan flags temporales y codificación cíclica para hora y día. Esto evita tratar 23:00 y 00:00 como valores artificialmente lejanos.

In [ ]:
day_map = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
df_features['hour'] = pd.to_numeric(df_features['hour_local'], errors='coerce').fillna(df_features['tx_datetime'].dt.hour).astype(int)
df_features['day_num'] = df_features['day_of_week'].map(day_map).fillna(df_features['tx_datetime'].dt.dayofweek).astype(int)
df_features['is_weekend'] = df_features['day_num'].isin([5, 6])
df_features['is_night'] = df_features['hour'].between(0, 5) | df_features['hour'].between(22, 23)
df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24)
df_features['day_sin'] = np.sin(2 * np.pi * df_features['day_num'] / 7)
df_features['day_cos'] = np.cos(2 * np.pi * df_features['day_num'] / 7)

temporal_features = ['hour', 'day_num', 'is_weekend', 'is_night', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
display(df_features[temporal_features].describe(include='all').T)


## 7) Columnas excluidas y set final de variables

In [ ]:
constant_cols = [c for c in df_features.columns if df_features[c].nunique(dropna=False) <= 1]
id_regex = re.compile(r'(^id$|_id$|_id_|^id_|hash|pan|stan|reference|track2|authorization_code|account_id)', re.IGNORECASE)
id_like_cols = [c for c in df_features.columns if id_regex.search(c)]
high_cardinality_cols = [c for c in df.columns if df[c].nunique(dropna=False) / len(df) > 0.90]

post_event_cols = [
    'approved', 'response_description', 'DE39_response_code', 'DE38_authorization_code',
    'DE44_additional_response_data', 'DE54_additional_amounts', 'DE56_original_data',
]
raw_or_reference_cols = [
    'transaction_id', 'client_id', 'tx_datetime', target_col,
    'DE2_PAN', 'pan_masked', 'pan_hash', 'DE35_track2_data_masked',
    'DE37_retrieval_reference_number', 'DE11_STAN', 'DE43_card_acceptor_name_location',
    'DE7_transmission_datetime', 'DE12_local_time', 'DE13_local_date', 'DE15_settlement_date',
]

excluded_cols = sorted(set(constant_cols + id_like_cols + high_cardinality_cols + post_event_cols + raw_or_reference_cols))

selected_original_numeric = [
    'amount_usd', 'amount_local', 'amount_tx_currency', 'distance_from_home_km',
    'client_baseline_amount', 'DE4_amount_transaction', 'DE6_amount_cardholder_billing',
    'DE9_conversion_rate_billing', 'DE18_merchant_category_code', 'DE19_acquirer_country_code',
    'DE22_pos_entry_mode', 'DE23_card_seq_number', 'DE25_pos_condition_code',
    'DE49_currency_code_transaction', 'DE123_pos_data_code',
]
selected_original_categorical = [
    'client_segment', 'channel', 'card_brand', 'currency_tx_alpha', 'client_home_city',
    'DE52_pin_data_present', 'DE55_emv_data_present', 'DE60_pos_terminal_type',
]

engineered_feature_cols = (
    ['is_pos_attended', 'is_card_present_purchase', 'is_ecom', 'is_atm', 'is_moto', 'is_domestic', 'is_foreign_currency', 'is_international_pos', 'requires_pin', 'has_emv']
    + amount_features
    + velocity_features
    + novelty_features
    + risk_features
    + temporal_features
)

numeric_feature_cols = [c for c in selected_original_numeric + engineered_feature_cols if c in df_features.columns and c not in excluded_cols]
categorical_feature_cols = [c for c in selected_original_categorical if c in df_features.columns and c not in excluded_cols]
feature_cols = list(dict.fromkeys(numeric_feature_cols + categorical_feature_cols))

reference_cols = ['transaction_id', 'client_id', 'tx_datetime', 'channel', 'DE60_pos_terminal_type', target_col]
output_cols = list(dict.fromkeys(reference_cols + feature_cols))

display(pd.DataFrame({
    'grupo': ['feature_cols', 'numeric_feature_cols', 'categorical_feature_cols', 'excluded_cols', 'output_cols'],
    'cantidad': [len(feature_cols), len(numeric_feature_cols), len(categorical_feature_cols), len(excluded_cols), len(output_cols)],
}))
print('Primeras feature_cols:')
print(feature_cols[:40])
print('\nColumnas excluidas principales:')
print(excluded_cols[:80])


## 8) Validación exploratoria

Estas tablas revisan si las variables ordenan riesgo, especialmente en compras presenciales.

In [ ]:
feature_audit = pd.DataFrame({
    'dtype': df_features[feature_cols].dtypes.astype(str),
    'missing_pct': df_features[feature_cols].isna().mean().mul(100),
    'nunique': df_features[feature_cols].nunique(dropna=False),
}).sort_values(['missing_pct', 'nunique'], ascending=[False, False])

display(feature_audit.head(30).round(3))

assert df_features['tx_datetime'].is_monotonic_increasing, 'El dataset no quedo ordenado temporalmente.'
first_txn_counts = df_features.groupby('client_id', sort=False)['client_txn_count_prev'].first()
assert first_txn_counts.eq(0).all(), 'La primera transaccion de cada cliente debe tener client_txn_count_prev = 0.'
assert not set(feature_cols).intersection(post_event_cols), 'Hay columnas de post-evento dentro de feature_cols.'

print('Checks de no leakage basico completados.')


In [ ]:
def decile_lift_table(frame, feature, target=target_col, q=10):
    tmp = frame[[feature, target]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp.empty or tmp[feature].nunique() < 3:
        return pd.DataFrame()
    tmp['decile'] = pd.qcut(tmp[feature], q=q, duplicates='drop')
    out = tmp.groupby('decile', observed=False)[target].agg(total='count', frauds='sum', fraud_rate='mean').reset_index()
    out['fraud_rate_pct'] = out['fraud_rate'] * 100
    out['share_data_pct'] = out['total'] / len(tmp) * 100
    return out.sort_values('fraud_rate', ascending=False)

lift_features = [
    'amount_vs_client_baseline', 'amount_zscore_customer', 'amount_vs_client_mean_prev',
    'time_since_last_txn_min', 'txn_count_prev_24h', 'amount_sum_prev_24h',
    'DE18_merchant_category_code_fraud_rate_prev_smooth', 'DE22_pos_entry_mode_fraud_rate_prev_smooth',
    'amount_vs_mcc_mean_prev', 'amount_vs_merchant_mean_prev',
]

for col in [c for c in lift_features if c in df_features.columns]:
    print(f'\n=== {col} | dataset completo ===')
    display(decile_lift_table(df_features, col).round(4))
    print(f'=== {col} | solo compras presenciales ===')
    display(decile_lift_table(df_features.loc[pos_eval_mask], col).round(4))


In [ ]:
pos_signal_rows = []
for col in [c for c in lift_features if c in df_features.columns]:
    table = decile_lift_table(df_features.loc[pos_eval_mask], col)
    if table.empty:
        continue
    pos_signal_rows.append({
        'feature': col,
        'top_decile_fraud_rate_pct': table.iloc[0]['fraud_rate_pct'],
        'bottom_decile_fraud_rate_pct': table.iloc[-1]['fraud_rate_pct'],
        'lift_top_vs_bottom': table.iloc[0]['fraud_rate'] / max(table.iloc[-1]['fraud_rate'], 1e-9),
        'top_decile_total': table.iloc[0]['total'],
        'top_decile_frauds': table.iloc[0]['frauds'],
    })

pos_signal_summary = pd.DataFrame(pos_signal_rows).sort_values('lift_top_vs_bottom', ascending=False)
display(pos_signal_summary.round(3))


## 9) Guardado del dataset transformado

In [ ]:
output_path = Path('../data/bolivia_features_obj_a.csv.gz') if Path('../data').exists() else Path('data/bolivia_features_obj_a.csv.gz')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_output = df_features[output_cols].copy()
df_output.to_csv(output_path, index=False, compression='gzip')

print(f'Dataset transformado guardado en: {output_path.resolve()}')
print(f'Shape guardado: {df_output.shape}')